In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from ctgan import CTGAN
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import pywt
from scipy import signal
from scipy.ndimage import gaussian_filter1d

# 1. 数据加载和预处理
df = pd.read_excel('../data/CTGAN_grouped_data_multi.xlsx')
print(f"原始数据集大小: {df.shape}")

# 2. 增强版小波分解滤波 - 更强的平滑效果
def apply_enhanced_wavelet_filter(series, wavelet='db4', level=1, threshold_factor=0.1):
    """
    使用增强版小波分解对数据进行滤波，提供更强的平滑效果
    - 使用更高级的小波基函数
    - 增加分解层数
    - 提高阈值因子
    """
    # 小波分解
    coeffs = pywt.wavedec(series.values, wavelet, level=level)
    
    # 对高频系数应用更强的阈值剔除
    for i in range(1, len(coeffs)):
        # 针对不同层级使用递增的阈值因子
        level_factor = threshold_factor * (1 + 0.2 * i)  # 高频成分阈值逐级增大
        threshold = level_factor * np.max(np.abs(coeffs[i]))
        # 使用硬阈值处理以获得更平滑结果
        coeffs[i] = pywt.threshold(coeffs[i], threshold, mode='hard')
    
    # 重构信号
    filtered_data = pywt.waverec(coeffs, wavelet)
    
    # 确保长度匹配
    filtered_data = filtered_data[:len(series)]
    return pd.Series(filtered_data, index=series.index)

# 3. 小波分解+高斯平滑的组合方法
def apply_combined_smoothing(series, wavelet='db4', level=1, threshold_factor=0.1, gaussian_sigma=1.8, iterations=1):
    """
    结合小波分解和高斯平滑的方法，获得极佳的平滑效果
    可以指定迭代次数进行多次平滑
    """
    result = series.copy()
    
    for _ in range(iterations):
        # 先应用小波分解滤波
        wavelet_filtered = apply_enhanced_wavelet_filter(result, wavelet, level, threshold_factor)
        
        # 再应用高斯平滑
        result = pd.Series(
            gaussian_filter1d(wavelet_filtered.values, sigma=gaussian_sigma),
            index=series.index
        )
    
    return result

# 创建平滑后的数据框，首先复制原始数据，只对特定列进行平滑处理
smoothed_df = df.copy()

# 定义需要平滑的列
columns_to_smooth = [f'x{i}' for i in range(1, 30)] 

# 对指定的列应用平滑处理
for col in columns_to_smooth:
    if col in df.columns:
            # 对其他需要平滑的列应用标准平滑处理
            smoothed_df[col] = apply_combined_smoothing(df[col], iterations=1)

# 确认数据结构
print(f"处理后数据集大小: {smoothed_df.shape}")
print(f"原始列名: {list(df.columns)}")
print(f"处理后列名: {list(smoothed_df.columns)}")
print(f"平滑处理的列: {columns_to_smooth}")

# 保存处理后的数据
smoothed_df.to_excel('../data/pinghua_data_x_names.xlsx', index=False)
print("选定变量平滑处理后的数据已保存为 '../data/pinghua_data_x_names.xlsx'")

# 添加可视化部分：原始数据和平滑后数据的对比
# 特别关注x25-x29的最后400个数据点
plt.figure(figsize=(18, 15))
plt.style.use('seaborn-v0_8-darkgrid')

# 可视化对比的列
vis_columns = ['x25', 'x26', 'x27', 'x28', 'x29']

# 获取最后400个数据点
last_n = 400
x_values = np.arange(df.shape[0] - last_n, df.shape[0])

for i, col in enumerate(vis_columns):
    plt.subplot(len(vis_columns), 1, i+1)
    
    # 绘制原始数据
    plt.plot(x_values, df[col].values[-last_n:], 'b-', alpha=0.6, linewidth=1.0, label='原始数据')
    
    # 绘制平滑后数据
    plt.plot(x_values, smoothed_df[col].values[-last_n:], 'r-', linewidth=1.5, label='平滑后数据')
    
    plt.title(f'{col} 变量平滑效果对比 (最后{last_n}个数据点)', fontsize=12)
    plt.legend(loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig('../data/smoothing_comparison_x25_x29.png', dpi=300, bbox_inches='tight')
plt.show()

# 计算平滑前后的差异统计量
print("\n平滑前后数据差异统计：")
for col in vis_columns:
    original_std = df[col].std()
    smoothed_std = smoothed_df[col].std()
    std_reduction = (original_std - smoothed_std) / original_std * 100
    
    # 计算平均绝对误差
    mae = np.mean(np.abs(df[col] - smoothed_df[col]))
    
    print(f"{col}: 标准差减少 {std_reduction:.2f}%, 平均绝对误差: {mae:.4f}")